In [55]:
# import packages

import os
import itertools
import shutil
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import CoolProp.CoolProp as CP

PROJECT_ROOT = os.path.abspath("../..")

In [56]:
# result-file naming configuration

CASE_ID = 'C0004'
RESULT_BASENAME = 'C0004R02_PARK25_AC_1D_SCAN_power'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'out', CASE_ID)
os.makedirs(OUTPUT_DIR, exist_ok=True)
RESULT_XLSX_NAME = f'{RESULT_BASENAME}.xlsx'

print(f'Result basename: {RESULT_BASENAME}')
print(f'Official Excel result: {RESULT_XLSX_NAME}')

Result basename: C0004R02_PARK25_AC_1D_SCAN_power
Official Excel result: C0004R02_PARK25_AC_1D_SCAN_power.xlsx


In [57]:
# model settings and parameter-scan ranges

SIM_DURATION_S = 1920.0
dt = 1.0
NUM_STEPS = int(round(SIM_DURATION_S / dt))
if not np.isclose(NUM_STEPS * dt, SIM_DURATION_S):
    raise ValueError('SIM_DURATION_S must be an integer multiple of dt.')
time_s = np.arange(NUM_STEPS, dtype=float) * dt

BATTERY_GEOMETRY = {'D_bat': 18e-3, 'H_bat': 65e-3}
MODULE_LAYOUT = {'N_r': 16, 'N_c': 20}
AIR_SETTINGS = {'fluid': 'Air', 'p_air': 101325.0, 'fan_efficiency': 0.35, 'C_rec': 0.30, 'ST_over_D_power': 1.25, 'chi': 1.0}

mission_stages = [
    ('takeoff', 0.0, 6.0),
    ('climb', 6.0, 36.0),
    ('transition1', 36.0, 180.0),
    ('cruise', 180.0, 1560.0),
    ('transition2', 1560.0, 1704.0),
    ('descent', 1704.0, 1734.0),
    ('hover', 1734.0, 1914.0),
    ('landing', 1914.0, 1920.0),
]

SCAN_VALUES = {'V_air_list': np.array([1.0, 2.0, 3.0, 4.0, 5.0, 7.5, 10.0], dtype=float), 'T_air_list': np.linspace(20.0, 40.0, 5), 's_gap_mm_list': np.linspace(1.0, 5.0, 5)}

D_bat = BATTERY_GEOMETRY['D_bat']
H_bat = BATTERY_GEOMETRY['H_bat']
N_r = MODULE_LAYOUT['N_r']
N_c = MODULE_LAYOUT['N_c']
fluid = AIR_SETTINGS['fluid']
p_air = AIR_SETTINGS['p_air']
fan_efficiency = AIR_SETTINGS['fan_efficiency']
C_rec = AIR_SETTINGS['C_rec']
ST_over_D_power = AIR_SETTINGS['ST_over_D_power']
chi = AIR_SETTINGS['chi']
V_air_list = SCAN_VALUES['V_air_list']
T_air_list = SCAN_VALUES['T_air_list']
s_gap_mm_list = SCAN_VALUES['s_gap_mm_list']

print(f'Mission duration: {SIM_DURATION_S:.1f} s')
print(f'Time step: {dt:.1f} s')
print(f'Number of time steps: {NUM_STEPS}')

Mission duration: 1920.0 s
Time step: 1.0 s
Number of time steps: 1920


In [58]:
# horizontal flight-speed profile used only for ram-pressure recovery

def get_horizontal_flight_velocity(t):
    V_190 = 190.0 / 3.6
    V_241 = 241.0 / 3.6

    if t < 6.0:
        V_x = 0.0
    elif t < 36.0:
        V_x = V_190 * (t - 6.0) / 30.0
    elif t < 180.0:
        V_x = V_190 + (V_241 - V_190) * (t - 36.0) / 144.0
    elif t < 1560.0:
        V_x = V_241
    elif t < 1704.0:
        V_x = V_241 - (V_241 - V_190) * (t - 1560.0) / 144.0
    elif t < 1734.0:
        V_x = V_190 * (1.0 - (t - 1704.0) / 30.0)
    else:
        V_x = 0.0

    return V_x

Vx_profile = np.array([get_horizontal_flight_velocity(t) for t in time_s], dtype=float)
print(f'Vx min/max: {Vx_profile.min():.3f} / {Vx_profile.max():.3f} m/s')

Vx min/max: 0.000 / 66.944 m/s


In [59]:
# build exactly the same parameter-scan combinations as the original notebook

def build_scan_cases(V_air_list, T_air_list, s_gap_mm_list):
    cases = []
    for case_count, (V_air, T_air, s_gap_mm) in enumerate(itertools.product(V_air_list, T_air_list, s_gap_mm_list), start=1):
        cases.append({'case_id': f'{CASE_ID}_S{case_count:03d}', 'V_air': float(V_air), 'T_air': float(T_air), 's_gap_mm': float(s_gap_mm)})
    return cases, pd.DataFrame(cases)

scan_cases, scan_cases_df = build_scan_cases(V_air_list, T_air_list, s_gap_mm_list)
print(f'Total cases: {len(scan_cases)}')
scan_cases_df.head()

Total cases: 175


,case_id,V_air,T_air,s_gap_mm
0,C0004_S001,1.0,20.0,1.0
1,C0004_S002,1.0,20.0,2.0
2,C0004_S003,1.0,20.0,3.0
3,C0004_S004,1.0,20.0,4.0
4,C0004_S005,1.0,20.0,5.0


In [60]:
# calculate only the air properties needed by the power model

air_props_cache = {}
for T_air in sorted(scan_cases_df['T_air'].unique()):
    T_air = float(T_air)
    T_air_K = T_air + 273.15
    mu_air = CP.PropsSI('V', 'T', T_air_K, 'P', p_air, fluid)
    rho_air = CP.PropsSI('D', 'T', T_air_K, 'P', p_air, fluid)
    air_props_cache[T_air] = {'mu_air': mu_air, 'rho_air': rho_air}

print(f'Air properties calculated for {len(air_props_cache)} inlet temperatures.')

Air properties calculated for 5 inlet temperatures.


In [61]:
# detailed fan supplemental-power model

def cal_air_fan_supplement_power_detailed(rho_air, mu_air, V_air_in, V_x, D_bat, N_c, A_in, fan_efficiency, C_rec=0.30, ST_over_D=1.25, chi=1.0):
    if rho_air <= 0.0:
        raise ValueError('rho_air must be positive.')
    if mu_air <= 0.0:
        raise ValueError('mu_air must be positive.')
    if V_air_in < 0.0:
        raise ValueError('V_air_in must be non-negative.')
    if D_bat <= 0.0:
        raise ValueError('D_bat must be positive.')
    if N_c <= 0:
        raise ValueError('N_c must be positive.')
    if A_in <= 0.0:
        raise ValueError('A_in must be positive.')
    if not 0.0 < fan_efficiency <= 1.0:
        raise ValueError('fan_efficiency must be in (0, 1].')
    if ST_over_D <= 1.0:
        raise ValueError('ST_over_D must be greater than 1.0.')

    S_T = ST_over_D * D_bat
    V_max = S_T / (S_T - D_bat) * V_air_in
    # Reynolds number for the battery-array pressure-drop correlation.
    # Use the local maximum velocity between cells.
    Re_D = rho_air * V_max * D_bat / mu_air
    f_park = 265.4245 * Re_D ** (-1.0482) + 0.0807
    dynamic_pressure_max = 0.5 * rho_air * V_max ** 2
    delta_p_battery = N_c * f_park * chi * dynamic_pressure_max
    delta_p_required = delta_p_battery
    delta_p_ram = C_rec * 0.5 * rho_air * V_x ** 2
    delta_p_fan = max(0.0, delta_p_required - delta_p_ram)
    V_dot_air = A_in * V_air_in
    m_dot_air = rho_air * V_dot_air
    P_air = delta_p_fan * V_dot_air
    P_fan = P_air / fan_efficiency

    return {'S_T_m': S_T, 'V_max_m_s': V_max, 'Re_D': Re_D, 'f_park': f_park, 'dynamic_pressure_max_Pa': dynamic_pressure_max, 'delta_p_battery_Pa': delta_p_battery, 'delta_p_required_Pa': delta_p_required, 'delta_p_ram_Pa': delta_p_ram, 'delta_p_fan_Pa': delta_p_fan, 'V_dot_air_m3_s': V_dot_air, 'm_dot_air_kg_s': m_dot_air, 'P_air_W': P_air, 'P_fan_W': P_fan}

print('Detailed fan-power function is ready.')

Detailed fan-power function is ready.


In [62]:
# run one power-only parameter-scan case

def run_one_case(case_id, V_air, T_air, s_gap_mm):
    air_props = air_props_cache[float(T_air)]
    rho_air = air_props['rho_air']
    mu_air = air_props['mu_air']

    s_gap = s_gap_mm * 1e-3
    S_T_scan = D_bat + s_gap

    S_T_power = ST_over_D_power * D_bat
    W_module_power = D_bat + (N_r - 1) * S_T_power
    A_in_power = W_module_power * H_bat

    P_fan_profile = np.zeros(NUM_STEPS)

    first_result = None
    for k in range(NUM_STEPS):
        result = cal_air_fan_supplement_power_detailed(rho_air=rho_air, mu_air=mu_air, V_air_in=V_air, V_x=Vx_profile[k], D_bat=D_bat, N_c=N_c, A_in=A_in_power, fan_efficiency=fan_efficiency, C_rec=C_rec, ST_over_D=ST_over_D_power, chi=chi)
        if first_result is None:
            first_result = result
        P_fan_profile[k] = result['P_fan_W']

    E_fan_total_J = np.sum(P_fan_profile) * dt

    row = {
        'case_id': case_id,
        'V_air_m_s': V_air,
        'T_air_in_C': T_air,
        's_gap_mm': s_gap_mm,
        'S_T_scan_mm': S_T_scan * 1e3,
        'rho_air_kg_m3': rho_air,
        'mu_air_Pa_s': mu_air,
        'Re_D': first_result['Re_D'],
        'f_park': first_result['f_park'],
        'ST_over_D_power': ST_over_D_power,
        'S_T_power_mm': first_result['S_T_m'] * 1e3,
        'V_max_m_s': first_result['V_max_m_s'],
        'dynamic_pressure_max_Pa': first_result['dynamic_pressure_max_Pa'],
        'delta_p_battery_Pa': first_result['delta_p_battery_Pa'],
        'A_in_power_m2': A_in_power,
        'V_dot_air_m3_s': first_result['V_dot_air_m3_s'],
        'm_dot_air_kg_s': first_result['m_dot_air_kg_s'],
        'fan_efficiency': fan_efficiency,
        'C_rec': C_rec,
        'E_fan_total_J': E_fan_total_J,
    }

    for stage_name, t0, t1 in mission_stages:
        stage_mask = (time_s >= t0) & (time_s < t1)
        stage_power = P_fan_profile[stage_mask]
        row[f'P_fan_mean_{stage_name}_W'] = np.mean(stage_power)

    return row


In [63]:
# run all 175 cases and export the simplified power-only scan table

def format_excel_table(xlsx_path):
    wb = load_workbook(xlsx_path)
    ws = wb.active
    ws.title = 'Power scan'
    body_font = Font(name='Times New Roman', size=10)
    header_font = Font(name='Times New Roman', size=10, bold=True)
    alignment = Alignment(horizontal='center', vertical='center')
    thin = Side(style='thin')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for row in ws.iter_rows():
        for cell in row:
            cell.font = header_font if cell.row == 1 else body_font
            cell.alignment = alignment
            cell.border = border
            if cell.row > 1 and isinstance(cell.value, float):
                cell.number_format = '0.000000'

    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    for col_idx, column_cells in enumerate(ws.columns, start=1):
        max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 30)

    wb.save(xlsx_path)

summary_rows = []

for n, case in enumerate(scan_cases, start=1):
    print(f"Running {n}/{len(scan_cases)}: {case['case_id']}, V_air={case['V_air']} m/s, T_air={case['T_air']} degC, s_gap={case['s_gap_mm']} mm")
    summary_rows.append(run_one_case(**case))

parameter_scan_table = pd.DataFrame(summary_rows)

# Column order:
# 1) scan inputs
# 2) key intermediate quantities used by the power model
# 3) total fan energy
# 4) stage-average fan power only
base_columns = [
    'case_id',
    'V_air_m_s',
    'T_air_in_C',
    's_gap_mm',
    'S_T_scan_mm',
    'rho_air_kg_m3',
    'mu_air_Pa_s',
    'Re_D',
    'f_park',
    'ST_over_D_power',
    'S_T_power_mm',
    'V_max_m_s',
    'dynamic_pressure_max_Pa',
    'delta_p_battery_Pa',
    'A_in_power_m2',
    'V_dot_air_m3_s',
    'm_dot_air_kg_s',
    'fan_efficiency',
    'C_rec',
    'E_fan_total_J',
]

stage_columns = []

for stage_name, _, _ in mission_stages:
    stage_columns.append(f'P_fan_mean_{stage_name}_W')

parameter_scan_table = parameter_scan_table[base_columns + stage_columns]

xlsx_path = os.path.join(OUTPUT_DIR, RESULT_XLSX_NAME)
parameter_scan_table.to_excel(xlsx_path, index=False)
format_excel_table(xlsx_path)

print(f'Saved official Excel: {xlsx_path}')
print(f'Rows exported: {len(parameter_scan_table)}')
print(f'Columns exported: {len(parameter_scan_table.columns)}')

data_dir = os.path.join(PROJECT_ROOT, 'data', CASE_ID)
os.makedirs(data_dir, exist_ok=True)
shutil.copy(xlsx_path, data_dir)
print(f'Copied scan result to data directory: {data_dir}')

parameter_scan_table.head()


Running 1/175: C0004_S001, V_air=1.0 m/s, T_air=20.0 degC, s_gap=1.0 mm
Running 2/175: C0004_S002, V_air=1.0 m/s, T_air=20.0 degC, s_gap=2.0 mm
Running 3/175: C0004_S003, V_air=1.0 m/s, T_air=20.0 degC, s_gap=3.0 mm
Running 4/175: C0004_S004, V_air=1.0 m/s, T_air=20.0 degC, s_gap=4.0 mm
Running 5/175: C0004_S005, V_air=1.0 m/s, T_air=20.0 degC, s_gap=5.0 mm
Running 6/175: C0004_S006, V_air=1.0 m/s, T_air=25.0 degC, s_gap=1.0 mm
Running 7/175: C0004_S007, V_air=1.0 m/s, T_air=25.0 degC, s_gap=2.0 mm
Running 8/175: C0004_S008, V_air=1.0 m/s, T_air=25.0 degC, s_gap=3.0 mm
Running 9/175: C0004_S009, V_air=1.0 m/s, T_air=25.0 degC, s_gap=4.0 mm
Running 10/175: C0004_S010, V_air=1.0 m/s, T_air=25.0 degC, s_gap=5.0 mm
Running 11/175: C0004_S011, V_air=1.0 m/s, T_air=30.0 degC, s_gap=1.0 mm
Running 12/175: C0004_S012, V_air=1.0 m/s, T_air=30.0 degC, s_gap=2.0 mm
Running 13/175: C0004_S013, V_air=1.0 m/s, T_air=30.0 degC, s_gap=3.0 mm
Running 14/175: C0004_S014, V_air=1.0 m/s, T_air=30.0 degC, 

,case_id,V_air_m_s,T_air_in_C,s_gap_mm,S_T_scan_mm,rho_air_kg_m3,mu_air_Pa_s,Re_D,f_park,ST_over_D_power,...,C_rec,E_fan_total_J,P_fan_mean_takeoff_W,P_fan_mean_climb_W,P_fan_mean_transition1_W,P_fan_mean_cruise_W,P_fan_mean_transition2_W,P_fan_mean_descent_W,P_fan_mean_hover_W,P_fan_mean_landing_W
0,C0004_S001,1.0,20.0,1.0,19.0,1.204575,0.000018,5954.833609,0.110017,1.25,...,0.3,442.444898,2.187356,0.410998,0.0,0.0,0.0,0.338087,2.187356,2.187356
1,C0004_S002,1.0,20.0,2.0,20.0,1.204575,0.000018,5954.833609,0.110017,1.25,...,0.3,442.444898,2.187356,0.410998,0.0,0.0,0.0,0.338087,2.187356,2.187356
2,C0004_S003,1.0,20.0,3.0,21.0,1.204575,0.000018,5954.833609,0.110017,1.25,...,0.3,442.444898,2.187356,0.410998,0.0,0.0,0.0,0.338087,2.187356,2.187356
3,C0004_S004,1.0,20.0,4.0,22.0,1.204575,0.000018,5954.833609,0.110017,1.25,...,0.3,442.444898,2.187356,0.410998,0.0,0.0,0.0,0.338087,2.187356,2.187356
4,C0004_S005,1.0,20.0,5.0,23.0,1.204575,0.000018,5954.833609,0.110017,1.25,...,0.3,442.444898,2.187356,0.410998,0.0,0.0,0.0,0.338087,2.187356,2.187356
